In [ ]:
# Google Drive mount and project-root setup for Colab
from pathlib import Path
import os

PROJECT_DIR = Path('/content/drive/MyDrive/phase_conditioned_diffusion_policy')

try:
    from google.colab import drive
except ImportError:
    print(f'Not running in Google Colab; keeping current working directory: {Path.cwd()}')
else:
    drive.mount('/content/drive')
    if not PROJECT_DIR.exists():
        raise FileNotFoundError(
            f'Expected project directory not found: {PROJECT_DIR}\n'
            'Update PROJECT_DIR to the Google Drive folder that contains this repository.'
        )
    os.chdir(PROJECT_DIR)
    print(f'Current working directory: {Path.cwd()}')


In [ ]:
# Colab dependency setup
import sys
import subprocess

try:
    import google.colab  # noqa: F401
except ImportError:
    print("Not running in Google Colab; skipping dependency installation and using the current environment.")
else:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", "."])

    import gymnasium as gym
    import mujoco
    import minari
    import torch

    gym.make("Ant-v5").close()

    print(f"gymnasium={gym.__version__}")
    print(f"mujoco={mujoco.__version__}")
    print(f"minari={minari.__version__}")
    print(f"torch={torch.__version__}")
    print("Ant-v5 environment smoke check passed.")


# 04 — Phase Trajectory Conditioning

Step 4 학습 노트북입니다. 핵심 로직은 `pcdp/` 모듈에 있고 이 파일은 실행 순서와 실험 파라미터만 노출합니다.


## 1. Artifact paths


In [ ]:
from pcdp.paths import ARTIFACT_ROOT, DATA_DIR, CHECKPOINTS_DIR, FIGURES_DIR, ensure_artifact_dirs
ensure_artifact_dirs()
print(f'✓ artifact root: {ARTIFACT_ROOT}')


## 2. Imports + Config

In [ ]:
import torch

from pcdp.configs import get_experiment_config
from pcdp.experiment_plots import plot_action_chunks, plot_loss_curve
from pcdp.experiment_runner import (
    build_model,
    build_noise_scheduler,
    load_data_and_build_loaders,
    sample_frequency_variants,
    train_or_load_checkpoint,
)
from pcdp.phase import trajectory_offline_frequencies
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch {torch.__version__}, device={device}')

cfg = get_experiment_config('phase_trajectory')
train_cond_fn = cfg.resolve_train_cond_fn()
sample_cond_fn = cfg.resolve_sample_cond_fn()
print(f'Experiment config: {cfg.name} — {cfg.display_name}')


## 3. Data

In [ ]:
data, train_ds, val_ds, train_loader, val_loader = load_data_and_build_loaders(cfg, DATA_DIR)


## 4. Model + Scheduler

In [ ]:
model = build_model(cfg, data, device=device)
noise_scheduler, ns_config, NUM_INFERENCE_STEPS = build_noise_scheduler(cfg)
ema = cfg.build_ema(model)


## 5. Train or Load

In [ ]:
TRAIN = True
train_losses, val_log, best_ema_state, CKPT_PATH = train_or_load_checkpoint(
    train=TRAIN, cfg=cfg, model=model, ema=ema, noise_scheduler=noise_scheduler,
    train_loader=train_loader, val_loader=val_loader, checkpoints_dir=CHECKPOINTS_DIR, device=device,
)


## 6. Loss Curve

In [ ]:
plot_loss_curve(train_losses, val_log, FIGURES_DIR / cfg.artifacts.loss_plot_name, title=cfg.display_name)


## 7. Offline Phase-Trajectory Sensitivity

In [ ]:
freqs, labels = trajectory_offline_frequencies(data)
ep_obs = val_ds[0]['obs'].unsqueeze(0)
samples_by_freq = sample_frequency_variants(
    model, ema, ns_config, ep_obs, data, sample_cond_fn, freqs, labels,
    device=device, num_inference_steps=NUM_INFERENCE_STEPS, dt=cfg.evaluation.dt, seed=data['seed'],
)
plot_action_chunks(
    samples_by_freq, FIGURES_DIR / 'phase_trajectory_sensitivity.png',
    title='Trajectory DP — same obs, different phase trajectories', act_dim=data['ACT_DIM'],
)
